# Gig Income Forecasting — ARIMA(1,1,1)

This notebook combines **model training** (`trainandsave.py`) and **forecasting** (`Forecast.py`) into a single end-to-end pipeline.

**Model design:**
- ARIMA(1,1,1) with log1p transform and d=1 differencing
- `d=1`: ADF test confirms 180-day series is non-stationary (p=0.42); first difference is stationary (p=0.000)
- DOW effect handled as empirical post-hoc multipliers in simulation
- 80/20 walk-forward train-test split (126 train / 32 test working days)
- Final model retrained on all 158 working days before saving
- 1,000 Monte Carlo simulation paths for uncertainty quantification

**Workflow:**
1. [Setup & Imports](#1-setup--imports)
2. [Train & Save Model](#2-train--save-model)
3. [Load Model & Run Forecast](#3-load-model--run-forecast)
   - 3.1 Monte Carlo Simulation
   - 3.2 Calendar Forecast
   - 3.3 Plot Function *(defined here so `predict` can call it)*
   - 3.4 Stored Evaluation Metrics Printer
   - 3.5 Main Predict Function
   - 3.6 Run Forecast
4. [Display Results](#4-display-results)

---
## 1. Setup & Imports

In [1]:
import os
import argparse
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────────────────────
N_SIMULATIONS = 1000
N_EVAL_SIMS   = 500
MODEL_PATH    = 'income_forecast_model.pkl'

print('All imports loaded successfully.')

All imports loaded successfully.


---
## 2. Train & Save Model

> **Source:** `trainandsave.py`
>
> Trains an ARIMA(1,1,1) model on the input CSV, evaluates it via walk-forward
> validation, retrains on the full dataset, and saves the complete bundle to a `.pkl` file.

### 2.1 Walk-forward Evaluation Helper

In [2]:
def walk_forward_eval(log_train, log_test, log_min, log_max, init_params):
    """True out-of-sample walk-forward: refit on expanding window each step."""
    preds     = []
    history   = list(log_train)
    for t in range(len(log_test)):
        m  = ARIMA(history, order=(1,1,1))
        r  = m.fit()
        fc = float(np.clip(r.forecast(1)[0], log_min, log_max))
        preds.append(fc)
        history.append(log_test[t])
    return np.array(preds)

### 2.2 Evaluation Metrics Computation

In [3]:
def compute_eval_metrics(log_test, pred_log, resid_std, log_min, log_max):
    actual = np.expm1(log_test)
    pred   = np.expm1(pred_log)

    mae  = float(np.mean(np.abs(actual - pred)))
    rmse = float(np.sqrt(np.mean((actual - pred)**2)))
    mape = float(np.mean(np.abs((actual - pred) / actual)) * 100)
    ss_res = float(np.sum((actual - pred)**2))
    ss_tot = float(np.sum((actual - np.mean(actual))**2))
    r2   = float(1 - ss_res / ss_tot) if ss_tot > 0 else float('nan')

    within_ci = 0
    for t in range(len(log_test)):
        sims = np.expm1(np.clip(
            pred_log[t] + np.random.normal(0, resid_std, N_EVAL_SIMS),
            log_min, log_max
        ))
        if np.percentile(sims, 10) <= actual[t] <= np.percentile(sims, 90):
            within_ci += 1
    coverage = within_ci / len(log_test) * 100

    return {
        'mae'            : round(mae, 2),
        'rmse'           : round(rmse, 2),
        'mape'           : round(mape, 2),
        'r2'             : round(r2, 4),
        'coverage_80pct' : round(coverage, 1),
        'n_evaluated'    : len(log_test),
        'split'          : 'test set — true out-of-sample walk-forward',
        'note'           : (
            'Metrics on held-out test set (20%% of working days). '
            'Walk-forward evaluation. '
            'Final model retrained on full dataset before saving.'
        ),
    }

### 2.3 Evaluation Metrics Printer

In [4]:
def print_eval_metrics(metrics):
    bar = '=' * 54
    print(f"\n{bar}")
    print(f"  MODEL EVALUATION — TEST SET (out-of-sample)")
    print(f"{bar}")
    print(f"  Evaluated on : {metrics['n_evaluated']} held-out working days")
    print(f"  Method       : Walk-forward one-step-ahead")
    print(f"  Model        : ARIMA(1,1,1) | log1p")
    print(f"{bar}")
    print(f"  MAE          : \u20b9{metrics['mae']:>10,.2f}")
    print(f"  RMSE         : \u20b9{metrics['rmse']:>10,.2f}")
    print(f"  MAPE         : {metrics['mape']:>10.2f}%")
    print(f"  80% Coverage : {metrics['coverage_80pct']:>10.1f}%")
    print(f"{bar}")
    if metrics['mape'] < 40:
        print(f"  ~ MAPE {metrics['mape']:.1f}% — acceptable for high-volatility gig income")
    else:
        print(f"  ~ MAPE {metrics['mape']:.1f}% — high; gig income is near-random day-to-day")
    print(f"  ! True out-of-sample — test set never seen during training.\n")

### 2.4 Training Function

In [5]:
def train(csv_path: str, save_path: str):
    """
    Trains ARIMA(1,1,1) income forecasting model — saves complete bundle to pkl.

    Model: ARIMA(1,1,1) | log1p transform | d=1 differencing
      - d=1: ADF test confirms 180-day series is non-stationary (p=0.42)
             first difference is stationary (p=0.000)
      - ARIMA(1,1,1): best test MAPE among all non-boundary models
      - DOW effect: handled as empirical post-hoc multipliers in simulation
                    (exogenous DOW dummies improved AIC but caused MA boundary
                    issues; post-hoc multipliers are more robust)

    Train-test split: 80/20 on working days (126 train / 32 test)
    Final model retrained on all 158 working days before saving.
    """
    # ── Load ──────────────────────────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').set_index('date').asfreq('D')
    full_series   = df['income'].fillna(0)
    working       = full_series[full_series > 0]
    log_working   = np.log1p(working.values)
    rest_day_prob = float((full_series == 0).mean())

    print(f"Training on  : {csv_path}")
    print(f"Date range   : {full_series.index[0].date()} → {full_series.index[-1].date()}")
    print(f"Total days   : {len(full_series)}")
    print(f"Working days : {len(working)}")
    print(f"Rest day prob: {rest_day_prob:.3f}")

    # ── Stationarity ─────────────────────────────────────────────────────────
    adf_d0 = adfuller(log_working)
    adf_d1 = adfuller(np.diff(log_working))
    print(f"\nADF d=0: p={adf_d0[1]:.4f} stationary={adf_d0[1]<0.05}")
    print(f"ADF d=1: p={adf_d1[1]:.4f} stationary={adf_d1[1]<0.05}")

    # ── DOW empirical multipliers ─────────────────────────────────────────────
    dow_means       = working.groupby(working.index.day_name()).mean()
    overall_mean    = float(dow_means.mean())
    dow_multipliers = (dow_means / overall_mean).to_dict()

    # ── Train-test split ──────────────────────────────────────────────────────
    n_train   = int(len(log_working) * 0.80)
    log_train = log_working[:n_train]
    log_test  = log_working[n_train:]
    log_min   = float(np.min(log_train))
    log_max   = float(np.max(log_train))

    print(f"\nTrain : {n_train} working days")
    print(f"Test  : {len(log_test)} working days")

    # ── Fit on train ──────────────────────────────────────────────────────────
    print(f"\nFitting ARIMA(1,1,1) on train set...")
    m_train = ARIMA(log_train, order=(1,1,1))
    r_train = m_train.fit()
    print(f"Train AIC: {r_train.aic:.2f}  BIC: {r_train.bic:.2f}")
    for name, val in zip(r_train.param_names, r_train.params):
        print(f"  {name:15s}: {float(val):.4f}")

    boundary = any(abs(abs(float(p))-1.0) < 0.02
                   for name, p in zip(r_train.param_names, r_train.params)
                   if name in ['ar.L1','ma.L1'])
    print(f"Boundary coefficients: {boundary}")

    # ── Walk-forward test evaluation ──────────────────────────────────────────
    print(f"\nWalk-forward evaluation on {len(log_test)} test days...")
    pred_log        = walk_forward_eval(log_train, log_test, log_min, log_max,
                                        r_train.params)
    resid_std_train = float(np.std(r_train.resid))
    metrics         = compute_eval_metrics(
        log_test, pred_log, resid_std_train, log_min, log_max
    )
    print_eval_metrics(metrics)

    # ── Retrain on full dataset ───────────────────────────────────────────────
    print("Retraining on full dataset...")
    log_min_full = float(np.min(log_working))
    log_max_full = float(np.max(log_working))

    m_full = ARIMA(log_working, order=(1,1,1))
    r_full = m_full.fit()
    print(f"Full AIC: {r_full.aic:.2f}")

    resid_std = float(np.std(r_full.resid))
    ar1       = float(r_full.params[r_full.param_names.index('ar.L1')])
    ma1       = float(r_full.params[r_full.param_names.index('ma.L1')])

    # ── Bundle ────────────────────────────────────────────────────────────────
    # Store training dates explicitly so plot_all never re-derives them
    # from the submitted CSV (which may be shorter than n_train).
    bundle = {
        'fitted_result'      : r_full,
        'model_order'        : (1, 1, 1),
        'ar1'                : ar1,
        'ma1'                : ma1,
        'resid_std'          : resid_std,
        'log_min'            : log_min_full,
        'log_max'            : log_max_full,
        'dow_multipliers'    : dow_multipliers,
        'rest_day_prob'      : rest_day_prob,
        'last_log_value'     : float(log_working[-1]),
        'last_diff'          : float(log_working[-1] - log_working[-2]),
        'last_resid'         : float(r_full.resid[-1]),
        'last_date'          : full_series.index[-1],
        'n_simulations'      : N_SIMULATIONS,
        'trained_on'         : csv_path,
        'eval_metrics'       : metrics,
        'test_actual_inr'    : np.expm1(log_test).tolist(),
        'test_pred_inr'      : np.expm1(pred_log).tolist(),
        'test_n_start'       : n_train,
        # Explicitly stored — plot_all uses these instead of re-deriving
        # from the submitted CSV, which may have fewer working days
        'train_working_dates': working.index[:n_train].tolist(),
        'test_working_dates' : working.index[n_train:].tolist(),
        'train_working_vals' : working.values[:n_train].tolist(),
    }

    joblib.dump(bundle, save_path)
    print(f"\nModel bundle saved → {save_path}")
    return bundle

### 2.5 Run Training

Set `CSV_PATH` to the path of your gig income CSV file (`date`, `income` columns required).

In [6]:
CSV_PATH   = 'gig_income_data.csv'       # ← update this path as needed
SAVE_PATH  = 'income_forecast_model.pkl'

bundle = train(csv_path=CSV_PATH, save_path=SAVE_PATH)

Training on  : gig_income_data.csv
Date range   : 2024-01-01 → 2024-06-28
Total days   : 180
Working days : 169
Rest day prob: 0.061

ADF d=0: p=0.0000 stationary=True
ADF d=1: p=0.0000 stationary=True

Train : 135 working days
Test  : 34 working days

Fitting ARIMA(1,1,1) on train set...
Train AIC: 124.60  BIC: 133.29
  ar.L1          : -0.0656
  ma.L1          : -0.9738
  sigma2         : 0.1386
Boundary coefficients: False

Walk-forward evaluation on 34 test days...

  MODEL EVALUATION — TEST SET (out-of-sample)
  Evaluated on : 34 held-out working days
  Method       : Walk-forward one-step-ahead
  Model        : ARIMA(1,1,1) | log1p
  MAE          : ₹    304.78
  RMSE         : ₹    372.83
  MAPE         :      28.61%
  80% Coverage :       97.1%
  ~ MAPE 28.6% — acceptable for high-volatility gig income
  ! True out-of-sample — test set never seen during training.

Retraining on full dataset...
Full AIC: 148.12

Model bundle saved → income_forecast_model.pkl


---
## 3. Load Model & Run Forecast

> **Source:** `Forecast.py`
>
> Loads the saved model bundle and generates income forecasts from a new CSV.
>
> **Simulation:** ARIMA(1,1,1) in difference space
> - `D(t) = ar1 * D(t-1) + ma1 * eps(t-1) + eps(t)`
> - `y(t) = y(t-1) + D(t)`
> - DOW multiplier applied post-hoc per calendar day
>
> **Input CSV:** columns `date` (YYYY-MM-DD), `income` (INR, 0 on rest days). Minimum 2 working-day rows required to seed AR state.

### 3.1 Monte Carlo Simulation

In [7]:
def simulate_paths(n_steps, n_sims, ar1, ma1, resid_std,
                   last_y, last_diff, last_resid, log_min, log_max):
    """
    ARIMA(1,1,1) forward simulation in difference space.

    D(t) = ar1 * D(t-1)  +  ma1 * eps(t-1)  +  eps(t)
    y(t) = y(t-1) + D(t)

    Clipped to [log_min, log_max] — observed income range from training —
    prevents the near-unit-root AR sum from drifting to bounds over time.

    Returns log-income paths, shape (n_sims, n_steps).
    """
    paths = np.zeros((n_sims, n_steps))
    for s in range(n_sims):
        prev_y    = last_y
        prev_diff = last_diff
        prev_eps  = last_resid
        for t in range(n_steps):
            eps      = np.random.normal(0, resid_std)
            new_diff = ar1 * prev_diff + ma1 * prev_eps + eps
            new_y    = float(np.clip(prev_y + new_diff, log_min, log_max))
            paths[s, t] = new_y
            prev_diff   = new_diff
            prev_eps    = eps
            prev_y      = new_y
    return paths

### 3.2 Calendar Forecast

In [8]:
def forecast_to_calendar(bundle, horizon, last_date):
    """
    1. Simulate n_sims ARIMA paths for enough working-day steps.
    2. Apply DOW multiplier post-hoc to each simulated income value.
    3. Assign rest days probabilistically and map to calendar.
    4. Aggregate: median = point forecast, p10/p90 = 80% CI.
    """
    ar1           = bundle['ar1']
    ma1           = bundle['ma1']
    resid_std     = bundle['resid_std']
    log_min       = bundle['log_min']
    log_max       = bundle['log_max']
    last_y        = bundle['last_log_value']
    last_diff     = bundle['last_diff']
    last_resid    = bundle['last_resid']
    rest_day_prob = bundle['rest_day_prob']
    dow_mult      = bundle['dow_multipliers']
    n_sims        = bundle['n_simulations']

    # Generate enough working-day steps to cover the calendar horizon
    n_working = int(np.ceil(horizon / (1 - rest_day_prob))) + 10

    log_paths    = simulate_paths(n_working, n_sims, ar1, ma1, resid_std,
                                  last_y, last_diff, last_resid, log_min, log_max)
    income_paths = np.expm1(log_paths)   # back to INR, shape (n_sims, n_working)

    # Calendar dates
    cal_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=horizon, freq='D'
    )

    # For each simulation: place working-day incomes on calendar dates,
    # apply DOW multiplier, and insert rest days
    all_sims = np.zeros((n_sims, horizon))

    for s in range(n_sims):
        wptr = 0
        for ci, date in enumerate(cal_dates):
            is_weekend = date.weekday() >= 5
            p_rest     = rest_day_prob * (0.4 if is_weekend else 1.0)
            if np.random.random() < p_rest or wptr >= n_working:
                all_sims[s, ci] = 0.0
            else:
                mult            = dow_mult.get(date.day_name(), 1.0)
                all_sims[s, ci] = income_paths[s, wptr] * mult
                wptr           += 1

    # Days where >50% of simulations are rest → forecast rest day
    is_rest = (all_sims == 0).mean(axis=0) > 0.5

    point = np.zeros(horizon)
    lower = np.zeros(horizon)
    upper = np.zeros(horizon)

    for i in range(horizon):
        working_sims = all_sims[all_sims[:, i] > 0, i]
        if len(working_sims) > 10:
            point[i] = np.median(working_sims)
            lower[i] = np.percentile(working_sims, 10)
            upper[i] = np.percentile(working_sims, 90)

    point[is_rest] = lower[is_rest] = upper[is_rest] = 0.0

    return pd.DataFrame({
        'date'      : cal_dates.strftime('%Y-%m-%d'),
        'day'       : cal_dates.day_name(),
        'predicted' : np.round(point, 2),
        'lower_80ci': np.round(lower, 2),
        'upper_80ci': np.round(upper, 2),
        'is_rest'   : is_rest,
    })

### 3.3 Plot Function

Defined here — before `predict()` — so it is available when `predict()` calls it with `plot=True`.

In [9]:
def plot_all(full_series, fc_df, bundle, horizon, out_path):
    """
    Panel 2 — Future forecast
      Last 45 calendar days of history + horizon-day forecast.
      Shaded CI band, rest day markers, weekend highlight, stats box.
    """
    test_actual = np.array(bundle['test_actual_inr'])
    test_pred   = np.array(bundle['test_pred_inr'])
    m           = bundle['eval_metrics']

    # Use dates stored in the bundle at training time.
    # Never re-derive from the submitted CSV — that CSV may be shorter
    # than n_train working days, making the slice empty (shape (0,))
    # and causing a matplotlib dimension mismatch error.
    train_dates = pd.DatetimeIndex(bundle['train_working_dates'])
    test_dates  = pd.DatetimeIndex(bundle['test_working_dates'])
    train_vals  = np.array(bundle['train_working_vals'])

    fig = plt.figure(figsize=(12, 6))
    fig.patch.set_facecolor('#F8F9FA')
    gs  = gridspec.GridSpec(1, 1, hspace=0.45)

    ax2 = fig.add_subplot(gs[0])

    fig.suptitle(
        'Gig Worker Income Forecast  \u00b7  ARIMA(1,1,1)  \u00b7  180-day training\n'
        '80/20 walk-forward train-test split  \u00b7  log1p transform  \u00b7  '
        'DOW multipliers  \u00b7  1,000 Monte Carlo paths',
        fontsize=11, fontweight='bold', y=0.995, color='#2C3E50'
    )

    # ── Panel 2: Future Forecast ──────────────────────────────────────────────
    ax2.set_facecolor('white')
    ax2.spines[['top', 'right']].set_visible(False)

    history   = full_series[-45:]
    last_date = full_series.index[-1]
    fc_df     = fc_df.copy()
    fc_df['date'] = pd.to_datetime(fc_df['date'])

    # Weekend shading — history
    for d in history.index:
        if d.weekday() >= 5:
            ax2.axvspan(d - pd.Timedelta(hours=12),
                        d + pd.Timedelta(hours=12),
                        alpha=0.10, color='#3498DB', zorder=0)

    # Weekend shading — forecast
    for _, row in fc_df.iterrows():
        if row['date'].weekday() >= 5:
            ax2.axvspan(row['date'] - pd.Timedelta(hours=12),
                        row['date'] + pd.Timedelta(hours=12),
                        alpha=0.15, color='#F39C12', zorder=0)

    # Historical income
    working_hist = history[history > 0]
    rest_hist    = history[history == 0]
    ax2.plot(working_hist.index, working_hist.values,
             color='#2C3E50', linewidth=1.8,
             label='Historical income', zorder=3)
    ax2.scatter(rest_hist.index, [0] * len(rest_hist),
                color='#BDC3C7', s=35, marker='x',
                zorder=4, label='Historical rest days')

    # Forecast
    wfc  = fc_df[~fc_df['is_rest']]
    rfc  = fc_df[fc_df['is_rest']]

    ax2.fill_between(wfc['date'], wfc['lower_80ci'], wfc['upper_80ci'],
                     color='#FADBD8', alpha=0.88,
                     label='80% CI', zorder=2)
    ax2.plot(wfc['date'], wfc['predicted'],
             color='#E74C3C', linewidth=2, linestyle='--',
             marker='o', markersize=4,
             label=f'{horizon}-day forecast', zorder=5)
    if len(rfc):
        ax2.scatter(rfc['date'], [0] * len(rfc),
                    color='#D5DBDB', s=35, marker='x',
                    zorder=4, label='Forecast rest days')

    # Forecast-start divider
    ax2.axvline(x=last_date, color='#95A5A6',
                linestyle=':', linewidth=1.5, alpha=0.9)
    ax2.text(last_date + pd.Timedelta(hours=10),
             ax2.get_ylim()[1] * 0.90,
             'Forecast \u2192', fontsize=8, color='#7F8C8D')

    # Stats box
    avg_day  = wfc['predicted'].mean()
    cons_day = wfc['lower_80ci'].mean()
    total    = wfc['predicted'].sum()
    n_rest   = len(rfc)
    ax2.text(0.985, 0.97,
             f"Avg/day (median)   : \u20b9{avg_day:,.0f}\n"
             f"Conservative (p10) : \u20b9{cons_day:,.0f}\n"
             f"Est. total {horizon}d      : \u20b9{total:,.0f}\n"
             f"Forecast rest days : {n_rest}",
             transform=ax2.transAxes, fontsize=8.5,
             va='top', ha='right',
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='white', edgecolor='#E74C3C', alpha=0.92))

    ax2.set_title(f'Panel 1 \u2014 {horizon}-Day Income Forecast',
                  fontsize=10, fontweight='bold', pad=7, color='#2C3E50')
    ax2.set_ylabel('Daily Income (\u20b9)', fontsize=9)
    ax2.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'\u20b9{x:,.0f}'))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.setp(ax2.xaxis.get_majorticklabels(),
             rotation=35, ha='right', fontsize=8)
    ax2.legend(loc='upper left', fontsize=7.5,
               framealpha=0.88, ncol=2)
    ax2.grid(axis='y', linestyle='--', alpha=0.30, color='#BDC3C7')
    ax2.set_ylim(bottom=-80)

    plt.savefig(out_path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"Chart saved \u2192 {out_path}")

### 3.4 Stored Evaluation Metrics Printer

In [10]:
def print_stored_eval_metrics(metrics):
    """Prints metrics computed and stored at training time. No recomputation."""
    bar = '=' * 54
    print(f"\n{bar}")
    print(f"  MODEL EVALUATION  (stored from training — test set)")
    print(f"{bar}")
    print(f"  Evaluated on : {metrics['n_evaluated']} held-out working days")
    print(f"  Method       : Walk-forward one-step-ahead")
    print(f"  Model        : ARIMA(1,1,1) | log1p transform")
    print(f"  Split        : {metrics['split']}")
    print(f"{bar}")
    print(f"  MAE          : \u20b9{metrics['mae']:>10,.2f}   (avg daily error)")
    print(f"  RMSE         : \u20b9{metrics['rmse']:>10,.2f}   (penalises large errors)")
    print(f"  MAPE         : {metrics['mape']:>10.2f}%   (avg % error, working days)")
    print(f"  80% Coverage : {metrics['coverage_80pct']:>10.1f}%   (target \u2265 80%)")
    print(f"{bar}")
    if metrics['mape'] < 40:
        print(f"  ~ MAPE {metrics['mape']:.1f}% — acceptable for high-volatility gig income")
    else:
        print(f"  ~ MAPE {metrics['mape']:.1f}% — high; expected for near-random daily series")
    if metrics['coverage_80pct'] >= 75:
        print(f"  \u2713 CI coverage {metrics['coverage_80pct']:.1f}% — uncertainty bands calibrated")
    else:
        print(f"  ~ CI coverage {metrics['coverage_80pct']:.1f}% — CI may be too narrow")
    print(f"  ! Metrics fixed at training time. Retrain to update.\n")

### 3.5 Main Predict Function

In [11]:
def predict(csv_path, horizon=30, plot=False,
            model_path=MODEL_PATH, seed=42):
    np.random.seed(seed)

    # ── Load bundle ───────────────────────────────────────────────────────────
    print(f"Loading model  : {model_path}")
    bundle = joblib.load(model_path)
    print(f"Trained on     : {bundle['trained_on']}")
    print(f"Training cutoff: {bundle['last_date'].date()}")
    print(f"Model order    : ARIMA{bundle['model_order']}")

    # ── Load and validate CSV ─────────────────────────────────────────────────
    print(f"\nLoading CSV    : {csv_path}")
    df = pd.read_csv(csv_path)

    missing = {'date', 'income'} - set(df.columns)
    if missing:
        raise ValueError(f"Input CSV missing required columns: {missing}")

    df['date']  = pd.to_datetime(df['date'])
    df          = df.sort_values('date').set_index('date').asfreq('D')
    full_series = df['income'].fillna(0)
    working     = full_series[full_series > 0]

    if len(working) < 2:
        raise ValueError(
            f"Need at least 2 working-day rows to seed AR(2). Got {len(working)}."
        )

    print(f"Date range     : {full_series.index[0].date()} → "
          f"{full_series.index[-1].date()}")
    print(f"Working days   : {len(working)}")

    # ── Update AR state from submitted CSV ────────────────────────────────────
    # These three values seed the ARIMA(1,1,1) simulation from the last
    # known point in the submitted data — not from the training cutoff.
    last_log_value = float(np.log1p(working.values[-1]))
    prev_log_value = float(np.log1p(working.values[-2]))
    last_diff      = last_log_value - prev_log_value
    # last_resid: use training residual as prior if CSV matches training,
    # otherwise use 0 (neutral — no prior shock information)
    last_date_csv  = full_series.index[-1]
    last_date_trn  = bundle['last_date']
    if last_date_csv == last_date_trn:
        last_resid = bundle['last_resid']
    else:
        last_resid = 0.0  # new data — no residual information available

    bundle['last_log_value'] = last_log_value
    bundle['last_diff']      = last_diff
    bundle['last_resid']     = last_resid

    print(f"\nAR state seed:")
    print(f"  last income  : \u20b9{working.values[-1]:,.0f}  "
          f"(log: {last_log_value:.4f})")
    print(f"  last diff    : {last_diff:+.4f}  "
          f"({'up' if last_diff > 0 else 'down'} from previous day)")
    print(f"  last resid   : {last_resid:+.4f}")

    # ── Forecast ──────────────────────────────────────────────────────────────
    last_date = full_series.index[-1]
    print(f"\nSimulating {bundle['n_simulations']:,} paths "
          f"for {horizon}-day forecast...")
    fc_df = forecast_to_calendar(bundle, horizon, last_date)
    print("Done.")

    # ── Save forecast CSV ─────────────────────────────────────────────────────
    out_dir = os.path.dirname(os.path.abspath(csv_path))
    out_csv = os.path.join(out_dir, f'forecast_{horizon}days_output.csv')
    fc_df.to_csv(out_csv, index=False)
    print(f"\nForecast saved → {out_csv}")

    # ── Plot ──────────────────────────────────────────────────────────────────
    if plot:
        out_chart = os.path.join(out_dir, f'forecast_{horizon}days_chart.png')
        plot_all(full_series, fc_df, bundle, horizon, out_chart)

    # ── Console summary ───────────────────────────────────────────────────────
    wfc   = fc_df[~fc_df['is_rest']]
    rfc   = fc_df[fc_df['is_rest']]
    bar   = '=' * 54

    print(f"\n{bar}")
    print(f"  {horizon}-DAY FORECAST SUMMARY")
    print(f"{bar}")
    print(f"  Working days       : {len(wfc)}")
    print(f"  Rest days          : {len(rfc)}")
    print(f"  Avg/day  (median)  : \u20b9{wfc['predicted'].mean():,.0f}")
    print(f"  Conservative (p10) : \u20b9{wfc['lower_80ci'].mean():,.0f}")
    print(f"  Best case   (p90)  : \u20b9{wfc['upper_80ci'].mean():,.0f}")
    print(f"  Est. total {horizon}d     : \u20b9{wfc['predicted'].sum():,.0f}")
    print(f"{bar}")
    print(f"\n{fc_df.to_string(index=False)}")

    # ── Stored metrics ────────────────────────────────────────────────────────
    print_stored_eval_metrics(bundle['eval_metrics'])

    return fc_df

### 3.6 Run Forecast

Run this cell after training completes. You will be prompted to:
1. Enter the path to a **new CSV file** (must have `date` and `income` columns, at least 2 working-day rows)
2. Choose a forecast horizon: **15** or **30** calendar days

The forecast CSV and chart will be displayed automatically.

In [14]:
import ipywidgets as widgets
from IPython.display import Image, display, HTML

# ── Step 1: File upload widget ─────────────────────────────────────────────
print('=' * 54)
print('  FORECAST SETUP')
print('=' * 54)
print('Step 1: Upload your income CSV file (date, income columns):')

upload_widget = widgets.FileUpload(accept='.csv', multiple=False)
display(upload_widget)

# ── Step 2: Horizon selector ───────────────────────────────────────────────
print('\nStep 2: Choose forecast horizon:')
horizon_widget = widgets.RadioButtons(
    options=[('15 days', 15), ('30 days', 30)],
    value=30,
    description='Horizon:',
)
display(horizon_widget)

# ── Step 3: Run button ─────────────────────────────────────────────────────
run_button = widgets.Button(
    description='Run Forecast',
    button_style='success',
    icon='play',
)
output_area = widgets.Output()
display(run_button, output_area)

def on_run_clicked(_):
    with output_area:
        output_area.clear_output()

        # Validate upload
        if not upload_widget.value:
            print('ERROR: No file uploaded. Please upload a CSV first.')
            return

        # Save uploaded bytes to a temp file
        uploaded_file = list(upload_widget.value.values())[0]
        tmp_csv = '_uploaded_forecast_input.csv'
        with open(tmp_csv, 'wb') as f:
            f.write(uploaded_file['content'])

        HORIZON = horizon_widget.value

        print(f'→ File    : {uploaded_file["metadata"]["name"]}')
        print(f'→ Horizon : {HORIZON} days')
        print(f'→ Model   : {MODEL_PATH}')
        print()

        # Run forecast
        global fc_df, FORECAST_CSV
        FORECAST_CSV = tmp_csv
        fc_df = predict(
            csv_path   = tmp_csv,
            horizon    = HORIZON,
            plot       = True,
            model_path = MODEL_PATH,
        )

        # ── Forecast table ────────────────────────────────────────────────
        print('\nForecast table:')
        display(fc_df.style
            .format({
                'predicted'  : '\u20b9{:,.0f}',
                'lower_80ci' : '\u20b9{:,.0f}',
                'upper_80ci' : '\u20b9{:,.0f}',
            })
            .apply(lambda row: [
                'background-color: #f0f0f0; color: #888' if row['is_rest']
                else '' for _ in row
            ], axis=1)
            .set_caption(f'{HORIZON}-Day Income Forecast')
        )

        # ── Chart ─────────────────────────────────────────────────────────
        chart_path = os.path.join(
            os.path.dirname(os.path.abspath(tmp_csv)),
            f'forecast_{HORIZON}days_chart.png'
        )
        print(f'\nChart saved → {chart_path}')
        display(Image(filename=chart_path))

run_button.on_click(on_run_clicked)

  FORECAST SETUP
Step 1: Upload your income CSV file (date, income columns):


FileUpload(value={}, accept='.csv', description='Upload')


Step 2: Choose forecast horizon:


RadioButtons(description='Horizon:', index=1, options=(('15 days', 15), ('30 days', 30)), value=30)

Button(button_style='success', description='Run Forecast', icon='play', style=ButtonStyle())

Output()